In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json
from jaxcmr.summarize import (
    calculate_aic_weights,
    generate_t_p_matrices,
    summarize_parameters,
    winner_comparison_matrix,
    raw_winner_comparison_matrix
)

In [3]:
fit_tag = "50_set_likelihood_fixed_term_best_of_3"
fit_dir = "projects/TalmiEEG/results/fits/"
target_directory = "projects/TalmiEEG/"

data_names = [
    "TalmiEEG",
]

model_names = [
    "EEGStrength",
    "EEGStrengthMultiplicative",
    "WeirdCMRNoStop",
    "SimpleAttentionECMRNoStop",
    "2EEGAttentionECMRNoStop",
    "SimpleTwoLayerECMRNoStop",
    "2EEGSimpleTwoLayerECMRNoStop",
    "SimpleTwoLayerECMRNoStopMultiplicative",
    "SimpleAttentionECMRNoStopMultiplicative",
    "2EEGSimpleTwoLayerECMRNoStopMultiplicative",
]

model_titles = [
    "EEGStrength",
    "EEGStrengthMultiplicative",
    "WeirdCMRNoStop",
    "SimpleAttentionECMRNoStop",
    "2EEGAttentionECMRNoStop",
    "SimpleTwoLayerECMRNoStop",
    "2EEGSimpleTwoLayerECMRNoStop",
    "SimpleTwoLayerECMRNoStopMultiplicative",
    "SimpleAttentionECMRNoStopMultiplicative",
    "2EEGSimpleTwoLayerECMRNoStopMultiplicative",
]


query_parameters = [
    "encoding_drift_rate",
    "start_drift_rate",
    "recall_drift_rate",
    "shared_support",
    "item_support",
    "learning_rate",
    "primacy_scale",
    "primacy_decay",
    "stop_probability_scale",
    "stop_probability_growth",
    "choice_sensitivity",
]


In [4]:
run_tag = "Model_Comparison"

if not model_titles:
    model_titles = model_names.copy()

for data_name in data_names:
    print(f"### {data_name}\n")
    results = []

    for model_name, model_title in zip(model_names, model_titles):
        fit_path = os.path.join(fit_dir, f"{data_name}_{model_name}_{fit_tag}.json")

        with open(fit_path) as f:
            results.append(json.load(f))
            if "subject" not in results[-1]["fits"]:
                results[-1]["fits"]["subject"] = results[-1]["subject"]
            if "allow_repeated_recalls" not in results[-1]["fixed"]:
                results[-1]["fixed"]["allow_repeated_recalls"] = False
                results[-1]["fits"]["allow_repeated_recalls"] = [False] * len(
                    results[-1]["fits"]["subject"]
                )
            results[-1]["name"] = model_title
            if "mfc_trace_sensitivity" in results[-1]["free"]:
                results[-1]["free"]["repetition_orthogonality"] = results[-1]["free"][
                    "mfc_trace_sensitivity"
                ]
                results[-1]["fits"]["repetition_orthogonality"] = results[-1]["fits"][
                    "mfc_trace_sensitivity"
                ]
                results[-1]["free"].pop("mfc_trace_sensitivity")
                results[-1]["fits"].pop("mfc_trace_sensitivity")

    summary = summarize_parameters(
        results, query_parameters, include_std=True, include_ci=True
    )

    table_path = os.path.join(
        target_directory, "tables", f"{data_name}_{fit_tag}_{run_tag}_parameters.md"
    )
    with open(table_path, "w") as f:
        f.write(summary)
    print(summary)

    df_t, df_p = generate_t_p_matrices(results)

    print(df_p.to_markdown())
    print()

    aic_weights = calculate_aic_weights(results)

    with open(
        os.path.join(
            target_directory,
            "tables",
            f"{data_name}_{fit_tag}_{run_tag}_aic_weights.md",
        ),
        "w",
    ) as f:
        f.write(aic_weights.to_markdown())

    print(aic_weights.to_markdown())
    print()

    df_comparison = winner_comparison_matrix(results)

    with open(
        os.path.join(
            target_directory,
            "tables",
            f"{data_name}_{fit_tag}_{run_tag}_winner_ratios.md",
        ),
        "w",
    ) as f:
        f.write(df_comparison.to_markdown().replace(" nan ", "     "))

    print(df_comparison.to_markdown().replace(" nan ", "     "))
    print()

    df_comparison = raw_winner_comparison_matrix(results)

    with open(
        os.path.join(
            target_directory,
            "tables",
            f"{data_name}_{fit_tag}_{run_tag}_raw_winner_ratios.md",
        ),
        "w",
    ) as f:
        f.write(df_comparison.to_markdown().replace(" nan ", "     "))

    print(df_comparison.to_markdown().replace(" nan ", "     "))
    print()


### TalmiEEG

| Parameter | Statistic | EEGStrength | EEGStrengthMultiplicative | WeirdCMRNoStop | SimpleAttentionECMRNoStop | 2EEGAttentionECMRNoStop | SimpleTwoLayerECMRNoStop | 2EEGSimpleTwoLayerECMRNoStop | SimpleTwoLayerECMRNoStopMultiplicative | SimpleAttentionECMRNoStopMultiplicative | 2EEGSimpleTwoLayerECMRNoStopMultiplicative |
|---|---|---|---|---|---|---|---|---|---|---|---|
| fitness | mean | 214.45 +/- 16.58 | 214.47 +/- 16.60 | 215.01 +/- 15.63 | 211.71 +/- 16.37 | 210.91 +/- 16.45 | 212.87 +/- 16.19 | 211.69 +/- 16.35 | 212.74 +/- 16.23 | 211.99 +/- 16.34 | 211.18 +/- 16.37 |
|  | std | 49.78 | 49.85 | 46.91 | 49.14 | 49.38 | 48.59 | 49.08 | 48.72 | 49.05 | 49.13 |
|  | min | 115.80 | 116.10 | 130.91 | 113.72 | 114.19 | 119.21 | 116.36 | 117.62 | 113.81 | 114.14 |
|  | max | 317.59 | 319.21 | 318.62 | 318.51 | 318.02 | 318.66 | 318.63 | 318.79 | 318.70 | 318.59 |
| encoding drift rate | mean |  |  | 0.66 +/- 0.11 | 0.64 +/- 0.11 | 0.63 +/- 0.10 | 0.68 +/- 0.10 | 0.76 +/-